In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, roc_auc_score
from torch.nn import functional as F
from sklearn.model_selection import train_test_split

from domain_shift.core.config import settings
from domain_shift.CycleGAN.triplet_data_loader_val import get_data_loader
from domain_shift.CycleGAN.triplet_models_val import CycleGAN
from domain_shift.data_extraction.process_DRIAMS import DRIAMS_bin_to_df
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# Load the data
driams_1 = DRIAMS_bin_to_df(settings.DRIAMS_B_PATH)
driams_2 = DRIAMS_bin_to_df(settings.DRIAMS_C_PATH)
driams_3 = DRIAMS_bin_to_df(settings.DRIAMS_D_PATH)

In [ ]:
# Load Cyclegan
cycle_gan = CycleGAN()
cycle_gan.load_models_via_paths(
    Path("/home/dive001/Documents/Master/maldi-tof-domain-shift/models/top_2_B_C/generator_B_to_C.pth")
)
generator = cycle_gan.generator_1_to_2
generator.eval()

/home/dive001/Documents/Master/maldi-tof-domain-shift/domain_shift/CycleGAN/triplet_models_val.py:709: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.generator_1_to_2.lo

In [4]:
def generate_synthetic(driams: pd.DataFrame) -> torch.Tensor:
    synthetic_driams = []
    for index, data_anchor in driams.iterrows():
        binned_data = data_anchor["binned_6000"]

        # Convert to tensor if necessary
        if not isinstance(binned_data, torch.Tensor):
            binned_data = torch.tensor(binned_data, dtype=torch.float32)

        # Generate synthetic data
        output = generator(binned_data.unsqueeze(0).to("cuda")).cpu().detach()
        
        # Flatten properly and append
        synthetic_driams.append(output.view(output.size(0), -1))  

    # Stack tensors
    synthetic_driams = torch.cat(synthetic_driams)
    return synthetic_driams

# E. coli - Ceftriaxone

In [5]:
species = "Escherichia coli"
antibiotic = "Ceftriaxone"

In [6]:
# Filter by the species
filtered_driams_1 = driams_1[driams_1["species"] == species]
filtered_driams_2 = driams_2[driams_2["species"] == species]
filtered_driams_3 = driams_3[driams_3["species"] == species]

In [7]:
# Keep only binned_6000 and antibiotic columns
filtered_driams_1 = filtered_driams_1[[antibiotic, "binned_6000"]]
filtered_driams_1.columns = ["antibiotic", "binned_6000"]
filtered_driams_2 = filtered_driams_2[[antibiotic, "binned_6000"]]
filtered_driams_2.columns = ["antibiotic", "binned_6000"]
filtered_driams_3 = filtered_driams_3[[antibiotic, "binned_6000"]]
filtered_driams_3.columns = ["antibiotic", "binned_6000"]


In [8]:
filtered_driams_1["antibiotic"].isna().sum(), filtered_driams_2["antibiotic"].isna().sum(), filtered_driams_3["antibiotic"].isna().sum()

(np.int64(0), np.int64(11), np.int64(19))

In [9]:
filtered_driams_1 = filtered_driams_1.dropna()
filtered_driams_2 = filtered_driams_2.dropna()
filtered_driams_3 = filtered_driams_3.dropna()

In [10]:
filtered_driams_1['antibiotic'].value_counts(), \
filtered_driams_2['antibiotic'].value_counts(), \
filtered_driams_3['antibiotic'].value_counts()

(antibiotic
 S    168
 R     45
 Name: count, dtype: int64,
 antibiotic
 S    765
 R    148
 I      3
 Name: count, dtype: int64,
 antibiotic
 S    1796
 R     198
 Name: count, dtype: int64)

In [11]:
# Remove rows with antibiotic == I
filtered_driams_1 = filtered_driams_1[filtered_driams_1["antibiotic"] != "I"]
filtered_driams_2 = filtered_driams_2[filtered_driams_2["antibiotic"] != "I"]
filtered_driams_3 = filtered_driams_3[filtered_driams_3["antibiotic"] != "I"]

filtered_driams_1['antibiotic'].value_counts(), \
filtered_driams_2['antibiotic'].value_counts(), \
filtered_driams_3['antibiotic'].value_counts()

(antibiotic
 S    168
 R     45
 Name: count, dtype: int64,
 antibiotic
 S    765
 R    148
 Name: count, dtype: int64,
 antibiotic
 S    1796
 R     198
 Name: count, dtype: int64)

In [12]:
synthetic_driams_1 = generate_synthetic(filtered_driams_1)
synthetic_driams_2 = generate_synthetic(filtered_driams_2)
synthetic_driams_3 = generate_synthetic(filtered_driams_3)

## Driams B

In [13]:
# Train a lightGBM classifier with driams_1 and test on driams_2 and driams 3
X = np.vstack(synthetic_driams_1)
y = filtered_driams_1["antibiotic"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lightgbm = LGBMClassifier(n_estimators=100, random_state=42)
lightgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 133, number of negative: 37
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052988 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 330907
[LightGBM] [Info] Number of data points in the train set: 170, number of used features: 5900
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.782353 -> initscore=1.279431
[LightGBM] [Info] Start training from score 1.279431
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

LGBMClassifier(random_state=42)

In [14]:
roc_auc_score(y_test, lightgbm.predict_proba(X_test)[:, 1])

np.float64(0.6142857142857143)

In [15]:
X_2 = np.vstack(synthetic_driams_2)
roc_auc_score(filtered_driams_2["antibiotic"], lightgbm.predict_proba(X_2)[:, 1])

np.float64(0.6453011835364777)

In [16]:
X_3 = np.vstack(synthetic_driams_3)
roc_auc_score(filtered_driams_3["antibiotic"], lightgbm.predict_proba(X_3)[:, 1])

np.float64(0.5804931272637286)

## Driams C

In [17]:
# Train a lightGBM classifier with driams_2 and test on driams_1 and driams 3
X = np.vstack(synthetic_driams_2)
y = filtered_driams_2["antibiotic"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lightgbm = LGBMClassifier(n_estimators=100, random_state=42)
lightgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 611, number of negative: 119
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.278666 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1391046
[LightGBM] [Info] Number of data points in the train set: 730, number of used features: 5999
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.836986 -> initscore=1.635973
[LightGBM] [Info] Start training from score 1.635973
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

LGBMClassifier(random_state=42)

In [18]:
roc_auc_score(y_test, lightgbm.predict_proba(X_test)[:, 1])

np.float64(0.6820420958351993)

In [19]:
X_1 = np.vstack(synthetic_driams_1)
roc_auc_score(filtered_driams_1["antibiotic"], lightgbm.predict_proba(X_1)[:, 1])

np.float64(0.6791005291005292)

In [20]:
X_3 = np.vstack(synthetic_driams_3)
roc_auc_score(filtered_driams_3["antibiotic"], lightgbm.predict_proba(X_3)[:, 1])

np.float64(0.5461322579919462)

## Driams D

In [21]:
# Train a lightGBM classifier with driams_2 and test on driams_1 and driams 3
X = np.vstack(synthetic_driams_3)
y = filtered_driams_3["antibiotic"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lightgbm = LGBMClassifier(n_estimators=100, random_state=42)
lightgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1430, number of negative: 165
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.452301 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1517727
[LightGBM] [Info] Number of data points in the train set: 1595, number of used features: 6000
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.896552 -> initscore=2.159484
[LightGBM] [Info] Start training from score 2.159484


LGBMClassifier(random_state=42)

In [22]:
roc_auc_score(y_test, lightgbm.predict_proba(X_test)[:, 1])

np.float64(0.7468123861566484)

In [23]:
X_1 = np.vstack(synthetic_driams_1)
roc_auc_score(filtered_driams_1["antibiotic"], lightgbm.predict_proba(X_1)[:, 1])

np.float64(0.6326719576719577)

In [24]:
X_2 = np.vstack(synthetic_driams_2)
roc_auc_score(filtered_driams_2["antibiotic"], lightgbm.predict_proba(X_2)[:, 1])

np.float64(0.6139992934110581)

# S. aureus - Oxacillin

In [25]:
species = "Staphylococcus aureus"
antibiotic = "Oxacillin"

In [26]:
# Filter by the species
filtered_driams_1 = driams_1[driams_1["species"] == species]
filtered_driams_2 = driams_2[driams_2["species"] == species]
filtered_driams_3 = driams_3[driams_3["species"] == species]

In [27]:
# Keep only binned_6000 and antibiotic columns
filtered_driams_1 = filtered_driams_1[[antibiotic, "binned_6000"]]
filtered_driams_1.columns = ["antibiotic", "binned_6000"]
filtered_driams_2 = filtered_driams_2[[antibiotic, "binned_6000"]]
filtered_driams_2.columns = ["antibiotic", "binned_6000"]
filtered_driams_3 = filtered_driams_3[[antibiotic, "binned_6000"]]
filtered_driams_3.columns = ["antibiotic", "binned_6000"]


In [28]:
filtered_driams_1["antibiotic"].isna().sum(), filtered_driams_2["antibiotic"].isna().sum(), filtered_driams_3["antibiotic"].isna().sum()

(np.int64(7), np.int64(0), np.int64(2174))

In [29]:
filtered_driams_1 = filtered_driams_1.dropna()
filtered_driams_2 = filtered_driams_2.dropna()
filtered_driams_3 = filtered_driams_3.dropna()

In [30]:
filtered_driams_1['antibiotic'].value_counts(), \
filtered_driams_2['antibiotic'].value_counts(), \
filtered_driams_3['antibiotic'].value_counts()

(antibiotic
 S    325
 R     21
 Name: count, dtype: int64,
 antibiotic
 S    697
 R     41
 Name: count, dtype: int64,
 Series([], Name: count, dtype: int64))

In [31]:
# Remove rows with antibiotic == I
filtered_driams_1 = filtered_driams_1[filtered_driams_1["antibiotic"] != "I"]
filtered_driams_2 = filtered_driams_2[filtered_driams_2["antibiotic"] != "I"]
filtered_driams_3 = filtered_driams_3[filtered_driams_3["antibiotic"] != "I"]

filtered_driams_1['antibiotic'].value_counts(), \
filtered_driams_2['antibiotic'].value_counts(), \
filtered_driams_3['antibiotic'].value_counts()

(antibiotic
 S    325
 R     21
 Name: count, dtype: int64,
 antibiotic
 S    697
 R     41
 Name: count, dtype: int64,
 Series([], Name: count, dtype: int64))

In [32]:
synthetic_driams_1 = generate_synthetic(filtered_driams_1)
synthetic_driams_2 = generate_synthetic(filtered_driams_2)

## Driams B

In [33]:
# Train a lightGBM classifier with driams_1 and test on driams_2 and driams 3
X = np.vstack(synthetic_driams_1)
y = filtered_driams_1["antibiotic"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lightgbm = LGBMClassifier(n_estimators=100, random_state=42)
lightgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 260, number of negative: 16
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.070140 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 532064
[LightGBM] [Info] Number of data points in the train set: 276, number of used features: 5932
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.942029 -> initscore=2.788093
[LightGBM] [Info] Start training from score 2.788093
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

LGBMClassifier(random_state=42)

In [34]:
roc_auc_score(y_test, lightgbm.predict_proba(X_test)[:, 1])

np.float64(0.7415384615384616)

In [35]:
X_2 = np.vstack(synthetic_driams_2)
roc_auc_score(filtered_driams_2["antibiotic"], lightgbm.predict_proba(X_2)[:, 1])

np.float64(0.5117052174825908)

## Driams C

In [36]:
# Train a lightGBM classifier with driams_2 and test on driams_1 and driams 3
X = np.vstack(synthetic_driams_2)
y = filtered_driams_2["antibiotic"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lightgbm = LGBMClassifier(n_estimators=100, random_state=42)
lightgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 557, number of negative: 33
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.219813 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1130581
[LightGBM] [Info] Number of data points in the train set: 590, number of used features: 5996
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.944068 -> initscore=2.826058
[LightGBM] [Info] Start training from score 2.826058
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

LGBMClassifier(random_state=42)

In [37]:
roc_auc_score(y_test, lightgbm.predict_proba(X_test)[:, 1])

np.float64(0.5026785714285714)

In [38]:
X_1 = np.vstack(synthetic_driams_1)
roc_auc_score(filtered_driams_1["antibiotic"], lightgbm.predict_proba(X_1)[:, 1])

np.float64(0.5010989010989011)